In [1]:
# recombine grade 6 validate and grade 8 validate into a 6+8 set with equal distribution

import pandas as pd
import json
import os

# load grade 6 and grade 8 datasets
def combine_and_split(grade1: int, grade2: int) -> None:
    with open (f'grade_{grade1}/val.json') as f:
        grade1_data = json.load(f)
    grade1_dataframe = pd.DataFrame(grade1_data)
    with open (f'grade_{grade2}/val.json') as f:
        grade2_data = json.load(f)
    grade2_dataframe = pd.DataFrame(grade2_data)

    grade1_dataframe['label'] = grade1
    grade2_dataframe['label'] = grade2


    combined = pd.concat([grade1_dataframe, grade2_dataframe], ignore_index=True)

    from sklearn.model_selection import train_test_split
    val1, val2 = train_test_split(combined, test_size=0.5, random_state=42, stratify=combined['label'])
    val1['label'].value_counts()
    try:
        os.mkdir(f'./grade_{grade1}_{grade2}')
    except:
        pass
    pd.DataFrame(val1).to_json(f'./grade_{grade1}_{grade2}/validate.json', orient='records', indent=2)
    pd.DataFrame(val2).to_json(f'./grade_{grade1}_{grade2}/test.json', orient='records', indent=2)

In [2]:
grade_mixes = [(3, 11)]

for mix in grade_mixes:
    combine_and_split(*mix)

In [ ]:

import datasets
import pandas as pd
import json
import os

from sklearn.model_selection import train_test_split
from huggingface_hub import login, HfApi
from datasets import load_dataset
token = os.getenv("HUGGINGFACE_HUB_TOKEN")
login(token=token)

# load grades {2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12} datasets
def combine_and_split_baseline(group) -> None:
    val_sets = []
    test_sets = []
    train_sets = []
    for grade in range(2, 13):
        # with open (f'grade_{grade}/val.json') as f:
        #     grade_data = json.load(f)
        dataset = load_dataset(f"williamplacroix/graded_wikilarge",
                                data_dir="cleaned/grade05",
                                split="train")

        train_dataframe = pd.DataFrame(dataset['train'])
        val_dataframe = pd.DataFrame(dataset['validate'])
        test_dataframe = pd.DataFrame(dataset['test'])
        #grade_dataframe = pd.DataFrame(dataset)
        train_dataframe['label'] = grade
        val_dataframe['label'] = grade
        test_dataframe['label'] = grade
        
        train_sets.append(train_dataframe)
        val_sets.append(val_dataframe)
        test_sets.append(test_dataframe)
        print(f"Grade {grade} - Train: {len(train_dataframe)}, Val: {len(val_dataframe)}, Test: {len(test_dataframe)}")

    combined_train = pd.concat(train_sets, ignore_index=True)
    combined_val = pd.concat(val_sets, ignore_index=True)
    combined_test = pd.concat(test_sets, ignore_index=True)

    train_val_sample, _ = train_test_split(combined_train, train_size=len(combined_val), random_state=42, stratify=combined_train['label'])
    train_test_sample, _ = train_test_split(combined_train, train_size=len(combined_test), random_state=42, stratify=combined_train['label'])
    
    val, _ = train_test_split(combined_val, train_size=512, random_state=42)#, stratify=train_val_sample['label'])
    test, _ = train_test_split(combined_test, train_size=512, random_state=42)#, stratify=train_test_sample['label'])
    print(val['label'].value_counts())
    print(test['label'].value_counts())
    print(val.shape)
    print(test.shape)
    # try:
    #     os.mkdir(f'./baseline')
    # except:
    #     pass 
    pd.DataFrame(val).to_json(f'comparative_datasets/{group}/baseline/validate.json', orient='records', indent=2)
    pd.DataFrame(test).to_json(f'comparative_datasets/{group}/baseline/test.json', orient='records', indent=2)
    # datasets.Dataset.from_pandas(val).push_to_hub("williamplacroix/wikilarge_baseline_val")
    # datasets.Dataset.from_pandas(test).push_to_hub("williamplacroix/wikilarge_baseline_test")
    # api = HfApi()
    # repo_id = f'williamplacroix/wikilarge_baseline_alpaca'
    # api.upload_file(
    #         path_or_fileobj='./data/wikilarge/cleaned_graded_splits/wikilarge_baseline_alpaca/validate.json',
    #         path_in_repo='validate.json',
    #         repo_id=repo_id,
    #         commit_message=f"Update validate split",
    #         repo_type="dataset"
    #     )
    # api.upload_file(
    #         path_or_fileobj='./data/wikilarge/cleaned_graded_splits/wikilarge_baseline_alpaca/test.json',
    #         path_in_repo='test.json',
    #         repo_id=repo_id,
    #         commit_message=f"Update test split",
    #         repo_type="dataset"
    #     )
    
for group in ['cleaned', 'original', 'augmented']:
    combine_and_split_baseline(group)    

Grade 2 - Train: 4832, Val: 512, Test: 512
Grade 3 - Train: 5984, Val: 512, Test: 512
Grade 4 - Train: 18016, Val: 512, Test: 512
Grade 5 - Train: 13856, Val: 512, Test: 512
Grade 6 - Train: 30080, Val: 512, Test: 512
Grade 7 - Train: 23552, Val: 512, Test: 512
Grade 8 - Train: 31456, Val: 512, Test: 512
Grade 9 - Train: 22016, Val: 512, Test: 512
Grade 10 - Train: 32576, Val: 512, Test: 512
Grade 11 - Train: 18016, Val: 512, Test: 512
Grade 12 - Train: 23584, Val: 512, Test: 512
label
8     58
12    58
3     54
7     54
5     47
11    45
9     43
6     40
4     39
10    38
2     36
Name: count, dtype: int64
label
8     58
12    58
3     54
7     54
5     47
11    45
9     43
6     40
4     39
10    38
2     36
Name: count, dtype: int64
(512, 4)
(512, 4)


In [ ]:
from datasets import load_dataset, concatenate_datasets
import random

# Replace with your actual dataset identifiers (name or dicts if configs needed)
grades = range(2, 13)

# Optionally: set the desired total size for final val/test splits
TARGET_VAL_SIZE = 512
TARGET_TEST_SIZE = 512

# Accumulators
datasets = []
train_sizes = []

# Load all datasets and compute train sizes
for grade in grades:
    print(f"Loading dataset for grade {grade}...")
    dataset = load_dataset(f"comparative_datasets/{group}/{grade}")  # or use dict if loading configs
    datasets.append(dataset)
    train_sizes.append(len(dataset['train']))


# Compute proportional weights based on train sizes
total_train = sum(train_sizes)
weights = [size / total_train for size in train_sizes]

# Sample proportionally from val/test splits
val_samples = []
test_samples = []

for ds, w in zip(datasets, weights):
    val_size = int(w * TARGET_VAL_SIZE)
    test_size = int(w * TARGET_TEST_SIZE)

    val_split = ds['validate']
    test_split = ds['test']

    # Shuffle and select proportionally
    val_sample = val_split.shuffle(seed=42).select(range(min(val_size, len(val_split))))
    test_sample = test_split.shuffle(seed=42).select(range(min(test_size, len(test_split))))

    val_samples.append(val_sample)
    test_samples.append(test_sample)

# Combine into unified datasets
val_combined = concatenate_datasets(val_samples)
test_combined = concatenate_datasets(test_samples)

print(f"Combined validation size: {len(val_combined)}")
print(f"Combined test size: {len(test_combined)}")


Loading dataset for grade 2...
Loading dataset for grade 3...
Loading dataset for grade 4...
Loading dataset for grade 5...
Loading dataset for grade 6...
Loading dataset for grade 7...
Loading dataset for grade 8...
Loading dataset for grade 9...
Loading dataset for grade 10...
Loading dataset for grade 11...
Loading dataset for grade 12...
Combined validation size: 506
Combined test size: 506
